# NVIDIA Nemotron Reasoning Challenge — Kaggle submission builder

This notebook runs the full pipeline (EDA → data prep → LoRA training → package) and produces **submission.zip**.

## Input dataset you must add

1. **Competition data** (required)  
   - Go to the [NVIDIA Nemotron Model Reasoning Challenge](https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge) page.
   - Click **"Add Data"** and add the **competition dataset** (often named like `nvidia-nemotron-model-reasoning-challenge`).  
   - It must contain **train.csv** (columns: `id`, `prompt`, `answer`) and **test.csv** (columns: `id`, `prompt`).

2. **Project code** (required)  
   - Either **Option A**: Push this project to a **public GitHub repo**, then set `GITHUB_REPO` in the next cell and the notebook will clone it.  
   - Or **Option B**: Create a **Kaggle Dataset** with your code: zip the project root (so the zip contains `scripts/`, `run_all.py`, `requirements.txt`). Add that dataset to the notebook; the notebook will look for it under `/kaggle/input/`.

## Settings

- **GPU**: Enable GPU (Settings → Accelerator → GPU) so LoRA training runs.
- **Internet**: Enable Internet if you use the GitHub clone option.
- **Secrets** (optional): To use real training data + CoT generation, add `ANTHROPIC_API_KEY` in Notebook Secrets and set `USE_SYNTHETIC_ONLY = False`.

In [ ]:
# ========== CONFIG (edit these) ==========
# Leave empty to auto-detect from /kaggle/input (first folder with train.csv + test.csv).
COMPETITION_INPUT_PATH = "/kaggle/input/datasets/sebmontreal/nvidia-nemotron-model-reasoning-challenge"  # e.g. "/kaggle/input/nvidia-nemotron-model-reasoning-challenge"

# GitHub repo URL (must be public). Leave empty if you use a Kaggle Dataset for code.
GITHUB_REPO = "https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge"

# Use only synthetic data (no API calls). Set False if you added ANTHROPIC_API_KEY in Secrets.
USE_SYNTHETIC_ONLY = True

# Skip evaluation step to save time (submission.zip does not need it).
SKIP_EVAL = True

# Use PEFT only for training (skip Unsloth). Set True on Kaggle to avoid "no kernel image" CUDA errors on newer GPUs (e.g. Blackwell).
USE_PEFT_ONLY = True

In [ ]:
# Suppress traitlets/papermill FutureWarning (Kaggle environment; safe to ignore)
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="traitlets")

## 1. Find competition data and prepare working dir

In [ ]:
import os
import shutil

WORKING = "/kaggle/working"
INPUT = "/kaggle/input"
DATA_DIR = os.path.join(WORKING, "data")
os.makedirs(DATA_DIR, exist_ok=True)

# Resolve competition data path
if COMPETITION_INPUT_PATH and os.path.isdir(COMPETITION_INPUT_PATH):
    comp_path = COMPETITION_INPUT_PATH
else:
    comp_path = None
    for name in os.listdir(INPUT):
        path = os.path.join(INPUT, name)
        if os.path.isdir(path):
            if os.path.isfile(os.path.join(path, "train.csv")) and os.path.isfile(os.path.join(path, "test.csv")):
                comp_path = path
                break
if comp_path is None:
    raise FileNotFoundError(
        "Competition data not found. Add the competition dataset (with train.csv and test.csv) to this notebook."
    )
print(f"Using competition data: {comp_path}")

for f in ["train.csv", "test.csv"]:
    src = os.path.join(comp_path, f)
    dst = os.path.join(DATA_DIR, f)
    shutil.copy2(src, dst)
    print(f"Copied {f}")

print(f"Data dir: {DATA_DIR}")
print(os.listdir(DATA_DIR))

## 2. Get project code (clone or copy from input)

In [ ]:
import subprocess

code_ready = False

if GITHUB_REPO and GITHUB_REPO.strip():
    # Clone from GitHub (enable Internet in notebook settings)
    repo = GITHUB_REPO.strip().rstrip("/")
    if not repo.endswith(".git"):
        repo = repo + ".git"
    print(f"Cloning {repo} into {WORKING}...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", repo, "repo_temp"],
        cwd=WORKING,
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        # Move contents to working (avoid nested folder); keep our data/
        for item in os.listdir(os.path.join(WORKING, "repo_temp")):
            if item == "data":
                continue
            src = os.path.join(WORKING, "repo_temp", item)
            dst = os.path.join(WORKING, item)
            if os.path.exists(dst):
                if os.path.isdir(dst):
                    shutil.rmtree(dst)
                else:
                    os.remove(dst)
            shutil.move(src, dst)
        shutil.rmtree(os.path.join(WORKING, "repo_temp"), ignore_errors=True)
        code_ready = True
        print("Clone done.")
    else:
        print("Clone failed:", result.stderr or result.stdout)

if not code_ready:
    # Look for code in an input dataset (zip or folder with scripts/)
    for name in os.listdir(INPUT):
        path = os.path.join(INPUT, name)
        if not os.path.isdir(path):
            continue
        scripts_path = os.path.join(path, "scripts", "01_eda.py")
        if os.path.isfile(scripts_path):
            print(f"Copying code from input dataset: {name}")
            for item in os.listdir(path):
                src = os.path.join(path, item)
                dst = os.path.join(WORKING, item)
                if item == "data":
                    continue
                if os.path.exists(dst):
                    if os.path.isdir(dst):
                        shutil.rmtree(dst)
                    else:
                        os.remove(dst)
                shutil.copytree(src, dst) if os.path.isdir(src) else shutil.copy2(src, dst)
            code_ready = True
            break
        # Try zip
        for f in os.listdir(path):
            if f.endswith(".zip"):
                zip_path = os.path.join(path, f)
                print(f"Unzipping {zip_path} to {WORKING}")
                shutil.unpack_archive(zip_path, WORKING)
                code_ready = os.path.isfile(os.path.join(WORKING, "scripts", "01_eda.py"))
                if code_ready:
                    break
        if code_ready:
            break

if not code_ready:
    raise FileNotFoundError(
        "Project code not found. Set GITHUB_REPO to your public repo URL, or add a Kaggle Dataset that contains the project (scripts/, run_all.py, requirements.txt)."
    )

# Ensure competition data is in place (in case clone/zip had a data/ folder)
for f in ["train.csv", "test.csv"]:
    src = os.path.join(comp_path, f)
    dst = os.path.join(DATA_DIR, f)
    if os.path.isfile(src):
        shutil.copy2(src, dst)

print("Project files:", os.listdir(WORKING))

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print("Requirements installed.")

## 4. Run pipeline

In [ ]:
import sys

args = ["python", "run_all.py"]
if USE_SYNTHETIC_ONLY:
    args.append("--synthetic-only")
if SKIP_EVAL:
    args.append("--skip-eval")
if USE_PEFT_ONLY:
    args.append("--peft-only")

print("Running:", " ".join(args))
result = subprocess.run(args, cwd=WORKING)
if result.returncode != 0:
    sys.exit(result.returncode)
print("Pipeline finished.")

## 5. Verify submission

In [ ]:
sub_path = os.path.join(WORKING, "submission.zip")
if os.path.isfile(sub_path):
    print(f"submission.zip size: {os.path.getsize(sub_path):,} bytes")
    print("You can submit this file to the competition.")
else:
    print("submission.zip not found. Check pipeline logs above.")